In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CMU-MOSI")

MANIFEST = ROOT / "preprocessing/shared/manifest.csv"
PILOT = ROOT / "preprocessing/shared/pilot_manifest.csv"
RAW_ZIP = ROOT / "Raw.zip"

for name, path in {
    "Dataset": ROOT,
    "Manifest": MANIFEST,
    "Pilot manifest": PILOT,
    "Raw ZIP": RAW_ZIP,
}.items():
    print(f"{name}: {'OK' if path.exists() else 'NOT FOUND'}")
    print(f"  {path}")


Dataset: OK
  /content/drive/MyDrive/CMU-MOSI
Manifest: OK
  /content/drive/MyDrive/CMU-MOSI/preprocessing/shared/manifest.csv
Pilot manifest: OK
  /content/drive/MyDrive/CMU-MOSI/preprocessing/shared/pilot_manifest.csv
Raw ZIP: OK
  /content/drive/MyDrive/CMU-MOSI/Raw.zip


In [ ]:
import pandas as pd

pilot = pd.read_csv(
    PILOT,
    dtype={
        "id": str,
        "video_id": str,
        "clip_id": str,
    }
)

print("Số mẫu pilot:", len(pilot))
print("Danh sách cột:", pilot.columns.tolist())

display(pilot.head())


Số mẫu pilot: 10
Danh sách cột: ['id', 'video_id', 'clip_id', 'split', 'raw_text', 'label', 'source_archive', 'video_relpath', 'video_size_bytes']


,id,video_id,clip_id,split,raw_text,label,source_archive,video_relpath,video_size_bytes
0,7JsX8y1ysxY$_$12,7JsX8y1ysxY,12,train,I DONT KNOW CLASSIC SUPERHERO KIND OF GETTING ...,0.6,Raw.zip,Raw/7JsX8y1ysxY/12.mp4,251905
1,8OtFthrtaJM$_$23,8OtFthrtaJM,23,train,BUT HES JUST THE CREEPIEST OLD GUY YOULL EVER ...,-1.0,Raw.zip,Raw/8OtFthrtaJM/23.mp4,368129
2,8d-gEyoeBzc$_$5,8d-gEyoeBzc,5,train,UM AND IM GONNA REALLY POINT OUT WHY THAT I DO...,-2.0,Raw.zip,Raw/8d-gEyoeBzc/5.mp4,190192
3,9T9Hf74oK10$_$9,9T9Hf74oK10,9,train,UM IN A WAY A LOT OF THE THEMES IN NEVER LET M...,1.6,Raw.zip,Raw/9T9Hf74oK10/9.mp4,272052
4,Bfr499ggo-0$_$12,Bfr499ggo-0,12,train,THEY SHOULDVE JUST CUT THE WEDDING IN THE BEGI...,-1.4,Raw.zip,Raw/Bfr499ggo-0/12.mp4,93286


In [ ]:
import zipfile
import shutil
from pathlib import Path

sample = pilot.iloc[0]

sample_id = sample["id"]
video_relpath = str(sample["video_relpath"])

WORK_DIR = Path("/content/audio_pilot")
VIDEO_DIR = WORK_DIR / "videos"
WAV_DIR = WORK_DIR / "wav"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
WAV_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(RAW_ZIP, "r") as z:
    normalized = video_relpath.replace("\\", "/").lstrip("./")

    matches = [
        name for name in z.namelist()
        if name.replace("\\", "/").lstrip("./") == normalized
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Tìm thấy {len(matches)} file cho: {video_relpath}"
        )

    video_path = VIDEO_DIR / f"pilot_video{Path(normalized).suffix}"

    with z.open(matches[0]) as source:
        with open(video_path, "wb") as target:
            shutil.copyfileobj(source, target)

print("Sample ID:", sample_id)
print("Video:", video_path)
print("Size:", video_path.stat().st_size, "bytes")


Sample ID: 7JsX8y1ysxY$_$12
Video: /content/audio_pilot/videos/pilot_video.mp4
Size: 251905 bytes


In [ ]:
!ffmpeg -version


In [ ]:
import subprocess

wav_path = WAV_DIR / "pilot_audio.wav"

command = [
    "ffmpeg",
    "-y",
    "-i", str(video_path),
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    str(wav_path)
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

print("Tách Audio thành công!")
print("File WAV:", wav_path)


Tách Audio thành công!
File WAV: /content/audio_pilot/wav/pilot_audio.wav


In [ ]:
import soundfile as sf

# Đọc thông tin file WAV
info = sf.info(str(wav_path))

print("Sample rate:", info.samplerate)
print("Channels:", info.channels)
print("Duration:", info.duration, "giây")
print("Subtype:", info.subtype)

# Kiểm tra cấu hình theo yêu cầu nhóm
assert info.samplerate == 16000
assert info.channels == 1
assert info.subtype == "PCM_16"
assert info.duration > 0

print("PASS: File WAV đúng cấu hình!")


Sample rate: 16000
Channels: 1
Duration: 5.2346875 giây
Subtype: PCM_16
PASS: File WAV đúng cấu hình!


In [ ]:
import shutil
from pathlib import Path

# Thư mục lưu WAV trên Google Drive
SAVE_DIR = ROOT / "preprocessing/audio/wav"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Đặt tên WAV theo ID trong manifest
save_path = SAVE_DIR / f"{sample_id}.wav"

# Sao chép file từ Colab sang Google Drive
shutil.copy2(wav_path, save_path)

print("ID:", sample_id)
print("Đường dẫn WAV:", save_path)
print("Tồn tại:", save_path.exists())
print("Kích thước:", save_path.stat().st_size, "bytes")


ID: 7JsX8y1ysxY$_$12
Đường dẫn WAV: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/wav/7JsX8y1ysxY$_$12.wav
Tồn tại: True
Kích thước: 167588 bytes


In [ ]:
import soundfile as sf

saved_info = sf.info(str(save_path))

print("Sample rate:", saved_info.samplerate)
print("Channels:", saved_info.channels)
print("Duration:", saved_info.duration, "giây")
print("Subtype:", saved_info.subtype)

assert saved_info.samplerate == 16000
assert saved_info.channels == 1
assert saved_info.subtype == "PCM_16"
assert saved_info.duration > 0

print("PASS: WAV trên Drive hợp lệ!")


Sample rate: 16000
Channels: 1
Duration: 5.2346875 giây
Subtype: PCM_16
PASS: WAV trên Drive hợp lệ!


In [ ]:
from pathlib import Path

# Thư mục dự án đã sử dụng ở bước trước
MODULE_DIR = ROOT / "src" / "preprocessing"

# Tạo thư mục nếu chưa tồn tại
MODULE_DIR.mkdir(parents=True, exist_ok=True)

# Đường dẫn file module Audio
module_path = MODULE_DIR / "audio_processor.py"

print("Thư mục module:", MODULE_DIR)
print("Thư mục tồn tại:", MODULE_DIR.exists())
print("Đường dẫn file Python:", module_path)


Thư mục module: /content/drive/MyDrive/CMU-MOSI/src/preprocessing
Thư mục tồn tại: True
Đường dẫn file Python: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/audio_processor.py


In [ ]:
code = '''
from pathlib import Path
import subprocess
import soundfile as sf


def prepare_audio(video_path, wav_path, config) -> Path:
    video_path = Path(video_path)
    wav_path = Path(wav_path)

    # 1. Kiểm tra video đầu vào
    if not video_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy video: {video_path}"
        )

    if video_path.stat().st_size == 0:
        raise ValueError("Video đầu vào rỗng")

    # 2. Tạo thư mục lưu WAV
    wav_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # 3. Tách audio bằng FFmpeg
    command = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-ac", "1",
        "-ar", str(config["sample_rate"]),
        "-c:a", "pcm_s16le",
        str(wav_path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"FFmpeg thất bại: {result.stderr}"
        )

    # 4. Kiểm tra WAV đầu ra
    info = sf.info(str(wav_path))

    if info.samplerate != config["sample_rate"]:
        raise ValueError("Sample rate không đúng")

    if info.channels != 1:
        raise ValueError("WAV không phải mono")

    if info.subtype != "PCM_16":
        raise ValueError("WAV không phải PCM 16-bit")

    if info.duration <= 0:
        raise ValueError("WAV có thời lượng không hợp lệ")

    # 5. Trả về đường dẫn WAV
    return wav_path
'''

# Ghi code vào file Python trên Google Drive
module_path.write_text(code, encoding="utf-8")

print("Đã tạo file:", module_path)
print("File tồn tại:", module_path.exists())


Đã tạo file: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/audio_processor.py
File tồn tại: True


In [ ]:
import sys

# Thư mục chứa package preprocessing
SRC_DIR = ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import hàm từ audio_processor.py
from preprocessing.audio_processor import prepare_audio

print("Import prepare_audio thành công!")


Import prepare_audio thành công!


In [ ]:
# Cấu hình Audio của nhóm
config = {
    "sample_rate": 16000
}

# Đường dẫn lưu WAV thử nghiệm
test_wav_path = WAV_DIR / "pilot_audio_test.wav"

# Gọi hàm từ audio_processor.py
result_path = prepare_audio(
    video_path=video_path,
    wav_path=test_wav_path,
    config=config
)

print("Tách WAV bằng module thành công!")
print("File WAV:", result_path)


Tách WAV bằng module thành công!
File WAV: /content/audio_pilot/wav/pilot_audio_test.wav


In [ ]:
import soundfile as sf

# Đọc thông tin WAV vừa tạo
info = sf.info(str(result_path))

print("Sample rate:", info.samplerate)
print("Channels:", info.channels)
print("Duration:", info.duration, "giây")
print("Subtype:", info.subtype)

# Kiểm tra cấu hình theo yêu cầu nhóm
assert info.samplerate == 16000
assert info.channels == 1
assert info.subtype == "PCM_16"
assert info.duration > 0

# So sánh thời lượng với WAV đầu tiên
original_info = sf.info(str(wav_path))

assert abs(info.duration - original_info.duration) < 0.001

print("PASS: Hàm prepare_audio hoạt động đúng!")


Sample rate: 16000
Channels: 1
Duration: 5.2346875 giây
Subtype: PCM_16
PASS: Hàm prepare_audio hoạt động đúng!


In [ ]:
from pathlib import Path

# Thư mục video tạm trên Colab
PILOT_VIDEO_DIR = Path("/content/audio_pilot/pilot_videos")

# Thư mục WAV trên Google Drive
PILOT_WAV_DIR = ROOT / "preprocessing/audio/wav"

PILOT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
PILOT_WAV_DIR.mkdir(parents=True, exist_ok=True)

print("Thư mục video:", PILOT_VIDEO_DIR)
print("Thư mục WAV:", PILOT_WAV_DIR)
print("Số mẫu cần xử lý:", len(pilot))


Thư mục video: /content/audio_pilot/pilot_videos
Thư mục WAV: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/wav
Số mẫu cần xử lý: 10


In [ ]:
import zipfile
import pandas as pd

# Đọc danh sách file có trong Raw.zip
with zipfile.ZipFile(RAW_ZIP, "r") as z:
    zip_names = {
        name.replace("\\", "/").lstrip("./")
        for name in z.namelist()
    }

# Kiểm tra từng video trong pilot
results = []

for _, row in pilot.iterrows():
    video_relpath = str(row["video_relpath"]).replace("\\", "/").lstrip("./")

    results.append({
        "id": row["id"],
        "video_relpath": video_relpath,
        "exists_in_zip": video_relpath in zip_names
    })

# Tạo bảng kết quả
check_df = pd.DataFrame(results)

display(check_df)

print(
    "Số video tìm thấy:",
    check_df["exists_in_zip"].sum(),
    "/",
    len(check_df)
)


,id,video_relpath,exists_in_zip
0,7JsX8y1ysxY$_$12,Raw/7JsX8y1ysxY/12.mp4,True
1,8OtFthrtaJM$_$23,Raw/8OtFthrtaJM/23.mp4,True
2,8d-gEyoeBzc$_$5,Raw/8d-gEyoeBzc/5.mp4,True
3,9T9Hf74oK10$_$9,Raw/9T9Hf74oK10/9.mp4,True
4,Bfr499ggo-0$_$12,Raw/Bfr499ggo-0/12.mp4,True
5,I5y0__X72p0$_$35,Raw/I5y0__X72p0/35.mp4,True
6,I5y0__X72p0$_$6,Raw/I5y0__X72p0/6.mp4,True
7,Nzq88NnDkEk$_$32,Raw/Nzq88NnDkEk/32.mp4,True
8,VbQk4H8hgr0$_$24,Raw/VbQk4H8hgr0/24.mp4,True
9,Vj1wYRQjB-o$_$2,Raw/Vj1wYRQjB-o/2.mp4,True


Số video tìm thấy: 10 / 10


In [ ]:
import zipfile
import shutil
import pandas as pd
import soundfile as sf

# Danh sách ghi nhận kết quả
audio_results = []

with zipfile.ZipFile(RAW_ZIP, "r") as z:

    for _, row in pilot.iterrows():
        clip_id = str(row["id"])
        video_relpath = str(row["video_relpath"]).replace("\\", "/").lstrip("./")

        video_file = PILOT_VIDEO_DIR / f"{clip_id}.mp4"
        wav_file = PILOT_WAV_DIR / f"{clip_id}.wav"

        try:
            # 1. Lấy video từ Raw.zip
            with z.open(video_relpath) as source:
                with open(video_file, "wb") as target:
                    shutil.copyfileobj(source, target)

            # 2. Tách WAV bằng module đã viết
            prepare_audio(
                video_path=video_file,
                wav_path=wav_file,
                config={"sample_rate": 16000}
            )

            # 3. Đọc thông tin WAV
            info = sf.info(str(wav_file))

            audio_results.append({
                "id": clip_id,
                "wav_path": str(wav_file),
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "subtype": info.subtype,
                "duration": info.duration,
                "status": "success",
                "error": ""
            })

            print(f"PASS: {clip_id}")

        except Exception as e:
            audio_results.append({
                "id": clip_id,
                "wav_path": str(wav_file),
                "sample_rate": None,
                "channels": None,
                "subtype": None,
                "duration": None,
                "status": "failed",
                "error": str(e)
            })

            print(f"FAIL: {clip_id} - {e}")

# Tạo bảng kết quả
audio_df = pd.DataFrame(audio_results)

display(audio_df)

print(
    "Thành công:",
    (audio_df["status"] == "success").sum(),
    "/",
    len(audio_df)
)


PASS: 7JsX8y1ysxY$_$12
PASS: 8OtFthrtaJM$_$23
PASS: 8d-gEyoeBzc$_$5
PASS: 9T9Hf74oK10$_$9
PASS: Bfr499ggo-0$_$12
PASS: I5y0__X72p0$_$35
PASS: I5y0__X72p0$_$6
PASS: Nzq88NnDkEk$_$32
PASS: VbQk4H8hgr0$_$24
PASS: Vj1wYRQjB-o$_$2


,id,wav_path,sample_rate,channels,subtype,duration,status,error
0,7JsX8y1ysxY$_$12,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,5.234687,success,
1,8OtFthrtaJM$_$23,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,8.671188,success,
2,8d-gEyoeBzc$_$5,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,5.606187,success,
3,9T9Hf74oK10$_$9,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,5.791938,success,
4,Bfr499ggo-0$_$12,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,2.587563,success,
5,I5y0__X72p0$_$35,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,2.192875,success,
6,I5y0__X72p0$_$6,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,3.957562,success,
7,Nzq88NnDkEk$_$32,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,3.353813,success,
8,VbQk4H8hgr0$_$24,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,2.494687,success,
9,Vj1wYRQjB-o$_$2,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,16000,1,PCM_16,4.816687,success,


Thành công: 10 / 10


In [ ]:
import subprocess
import json
import pandas as pd

def get_video_duration(video_path):
    command = [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True
    )

    data = json.loads(result.stdout)
    return float(data["format"]["duration"])


duration_results = []

for _, row in audio_df.iterrows():
    clip_id = row["id"]

    video_file = PILOT_VIDEO_DIR / f"{clip_id}.mp4"
    wav_file = PILOT_WAV_DIR / f"{clip_id}.wav"

    try:
        video_duration = get_video_duration(video_file)
        wav_duration = sf.info(str(wav_file)).duration

        difference = abs(video_duration - wav_duration)

        duration_results.append({
            "id": clip_id,
            "video_duration": video_duration,
            "wav_duration": wav_duration,
            "difference": difference,
            "status": "review" if difference > 0.1 else "pass"
        })

    except Exception as e:
        duration_results.append({
            "id": clip_id,
            "video_duration": None,
            "wav_duration": None,
            "difference": None,
            "status": "failed",
            "error": str(e)
        })

duration_df = pd.DataFrame(duration_results)

display(duration_df)

print("Đạt ngưỡng đối chiếu:",
      (duration_df["status"] == "pass").sum(),
      "/",
      len(duration_df))


,id,video_duration,wav_duration,difference,status
0,7JsX8y1ysxY$_$12,5.266667,5.234687,0.031980,pass
1,8OtFthrtaJM$_$23,8.700000,8.671188,0.028812,pass
2,8d-gEyoeBzc$_$5,5.633333,5.606187,0.027146,pass
3,9T9Hf74oK10$_$9,5.833333,5.791938,0.041395,pass
4,Bfr499ggo-0$_$12,2.633333,2.587563,0.045770,pass
5,I5y0__X72p0$_$35,2.200000,2.192875,0.007125,pass
6,I5y0__X72p0$_$6,3.966667,3.957562,0.009105,pass
7,Nzq88NnDkEk$_$32,3.400000,3.353813,0.046187,pass
8,VbQk4H8hgr0$_$24,2.533333,2.494687,0.038645,pass
9,Vj1wYRQjB-o$_$2,4.833333,4.816687,0.016646,pass


Đạt ngưỡng đối chiếu: 10 / 10


In [ ]:
import pandas as pd
import subprocess

# Thư mục lưu hồ sơ bàn giao
REPORT_DIR = ROOT / "preprocessing/audio/reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Bảng ánh xạ ID -> WAV và thông tin kiểm tra
wav_report = audio_df.merge(
    duration_df[
        ["id", "video_duration", "difference", "status"]
    ].rename(columns={"status": "duration_status"}),
    on="id",
    how="left",
    validate="one_to_one"
)

wav_report.to_csv(
    REPORT_DIR / "pilot_wav_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Nhật ký lỗi
error_log = wav_report[
    (wav_report["status"] != "success") |
    (wav_report["duration_status"] != "pass")
].copy()

error_log.to_csv(
    REPORT_DIR / "pilot_wav_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Ghi cấu hình và phiên bản FFmpeg
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True,
    check=True
).stdout.splitlines()[0]

config_text = f"""Audio preprocessing configuration
Tool: {ffmpeg_version}
Input: Original video from Raw.zip
Output: WAV
Sample rate: 16000 Hz
Channels: 1 (mono)
Codec: pcm_s16le (PCM 16-bit)
Silence trimming: No
Speed modification: No
FFmpeg arguments: -y -i INPUT -vn -ac 1 -ar 16000 -c:a pcm_s16le OUTPUT
Duration review threshold: 0.1 second (pilot check only)
"""

(REPORT_DIR / "pilot_wav_config.txt").write_text(
    config_text,
    encoding="utf-8"
)

print("Đã lưu hồ sơ tại:", REPORT_DIR)
print("1. pilot_wav_report.csv")
print("2. pilot_wav_errors.csv")
print("3. pilot_wav_config.txt")
print("Số mẫu:", len(wav_report))
print("Số mẫu cần kiểm tra thêm:", len(error_log))


Đã lưu hồ sơ tại: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/reports
1. pilot_wav_report.csv
2. pilot_wav_errors.csv
3. pilot_wav_config.txt
Số mẫu: 10
Số mẫu cần kiểm tra thêm: 0


In [ ]:
%pip install opensmile


In [ ]:
import opensmile
import importlib.metadata

version = importlib.metadata.version("opensmile")

print("Phiên bản openSMILE:", version)
print("Import openSMILE thành công!")


Phiên bản openSMILE: 2.6.0
Import openSMILE thành công!


In [ ]:
import opensmile

# Khởi tạo bộ trích xuất đặc trưng Audio
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors
)

print("Khởi tạo openSMILE thành công!")
print("Số chiều đặc trưng:", len(smile.feature_names))

print("\nDanh sách đặc trưng:")
for i, name in enumerate(smile.feature_names, start=1):
    print(f"{i}. {name}")


Khởi tạo openSMILE thành công!
Số chiều đặc trưng: 25

Danh sách đặc trưng:
1. Loudness_sma3
2. alphaRatio_sma3
3. hammarbergIndex_sma3
4. slope0-500_sma3
5. slope500-1500_sma3
6. spectralFlux_sma3
7. mfcc1_sma3
8. mfcc2_sma3
9. mfcc3_sma3
10. mfcc4_sma3
11. F0semitoneFrom27.5Hz_sma3nz
12. jitterLocal_sma3nz
13. shimmerLocaldB_sma3nz
14. HNRdBACF_sma3nz
15. logRelF0-H1-H2_sma3nz
16. logRelF0-H1-A3_sma3nz
17. F1frequency_sma3nz
18. F1bandwidth_sma3nz
19. F1amplitudeLogRelF0_sma3nz
20. F2frequency_sma3nz
21. F2bandwidth_sma3nz
22. F2amplitudeLogRelF0_sma3nz
23. F3frequency_sma3nz
24. F3bandwidth_sma3nz
25. F3amplitudeLogRelF0_sma3nz


In [ ]:
# Trích xuất đặc trưng từ WAV đầu tiên
features_df = smile.process_file(str(wav_path))

# Kiểm tra kết quả
print("Kích thước đặc trưng:", features_df.shape)

print("\n5 dòng đặc trưng đầu tiên:")
display(features_df.head())

print("\nKiểu dữ liệu:")
print(features_df.dtypes.value_counts())

print("\nTên các mức chỉ mục thời gian:")
print(features_df.index.names)


Kích thước đặc trưng: (519, 25)

5 dòng đặc trưng đầu tiên:


Loudness_sma3  \
file                                     start                  end                                     
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:00.020000       0.148436   
                                         0 days 00:00:00.010000 0 days 00:00:00.030000       0.153631   
                                         0 days 00:00:00.020000 0 days 00:00:00.040000       0.159224   
                                         0 days 00:00:00.030000 0 days 00:00:00.050000       0.159638   
                                         0 days 00:00:00.040000 0 days 00:00:00.060000       0.166636   

                                                                                        alphaRatio_sma3  \
file                                     start                  end                                       
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:00.020000       -12.826613   
                                         0 days 00:00:00.010000 0 days 00:00:00.030000       -10.854450   
                                         0 days 00:00:00.020000 0 days 00:00:00.040000       -10.266485   
                                         0 days 00:00:00.030000 0 days 00:00:00.050000        -9.687945   
                                         0 days 00:00:00.040000 0 days 00:00:00.060000       -10.348031   

                                                                                        hammarbergIndex_sma3  \
file                                     start                  end                                            
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:00.020000             26.076021   
                                         0 days 00:00:00.010000 0 days 00:00:00.030000             24.039450   
                                         0 days 00:00:00.020000 0 days 00:00:00.040000             21.107698   
                                         0 days 00:00:00.030000 0 days 00:00:00.050000             18.904755   
                                         0 days 00:00:00.040000 0 days 00:00:00.060000             18.928551   

                                                                                        slope0-500_sma3  \
file                                     start                  end                                       
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:00.020000        -0.027717   
                                         0 days 00:00:00.010000 0 days 00:00:00.030000        -0.017698   
                                         0 days 00:00:00.020000 0 days 00:00:00.040000        -0.015039   
                                         0 days 00:00:00.030000 0 days 00:00:00.050000        -0.021204   
                                         0 days 00:00:00.040000 0 days 00:00:00.060000        -0.026500   

                                                                                        slope500-1500_sma3  \
file                                     start                  end                                          
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:00.020000            0.011323   
                                         0 days 00:00:00.010000 0 days 00:00:00.030000            0.005636   
                                         0 days 00:00:00.020000 0 days 00:00:00.040000            0.000330   
                                         0 days 00:00:00.030000 0 days 00:00:00.050000            0.003129   
                                         0 days 00:00:00.040000 0 days 00:00:00.060000            0.004299   

                                                                                        spectralFlux_sma3  \
file                                     start                  end                                         
/content/audio_pilot/wav/pilot_audio.wav 0 days 00:00:00        0 days 00:00:0


Kiểu dữ liệu:
float32    25
Name: count, dtype: int64

Tên các mức chỉ mục thời gian:
['file', 'start', 'end']


In [ ]:
import numpy as np

# Chuyển bảng đặc trưng sang mảng NumPy
features = features_df.to_numpy(dtype=np.float32)

# Kiểm tra giá trị NaN và Inf
nan_count = np.isnan(features).sum()
inf_count = np.isinf(features).sum()

# Kiểm tra các giá trị hữu hạn
finite_mask = np.isfinite(features)

print("Shape:", features.shape)
print("Dtype:", features.dtype)
print("Số giá trị NaN:", nan_count)
print("Số giá trị Inf:", inf_count)
print("Số giá trị hữu hạn:", finite_mask.sum())
print("Tổng số giá trị:", features.size)

print("\nSố giá trị -201:", np.sum(features == -201.0))


Shape: (519, 25)
Dtype: float32
Số giá trị NaN: 0
Số giá trị Inf: 0
Số giá trị hữu hạn: 12975
Tổng số giá trị: 12975

Số giá trị -201: 1072


In [ ]:
import pandas as pd
import numpy as np

# Đếm số lần xuất hiện -201 ở từng đặc trưng
minus201_count = (features_df == -201.0).sum()

# Tạo bảng thống kê
minus201_report = pd.DataFrame({
    "feature_name": features_df.columns,
    "minus201_count": minus201_count.values,
    "percentage": (
        minus201_count.values / len(features_df) * 100
    )
})

# Chỉ hiển thị những đặc trưng có giá trị -201
minus201_report = minus201_report[
    minus201_report["minus201_count"] > 0
].sort_values("minus201_count", ascending=False)

display(minus201_report)

print("Tổng số giá trị -201:",
      minus201_report["minus201_count"].sum())


,feature_name,minus201_count,percentage
18,F1amplitudeLogRelF0_sma3nz,358,68.978805
21,F2amplitudeLogRelF0_sma3nz,357,68.786127
24,F3amplitudeLogRelF0_sma3nz,357,68.786127


Tổng số giá trị -201: 1072


In [ ]:
import numpy as np

# Khởi tạo mask từ các giá trị hữu hạn
observed_mask = np.isfinite(features)

# Chỉ áp dụng quy tắc -201 cho 3 đặc trưng đã xác định
floor_features = [
    "F1amplitudeLogRelF0_sma3nz",
    "F2amplitudeLogRelF0_sma3nz",
    "F3amplitudeLogRelF0_sma3nz"
]

feature_names = features_df.columns.tolist()

for name in floor_features:
    col_idx = feature_names.index(name)

    # -201 là giá trị sàn ở các đặc trưng tương đối F0
    observed_mask[:, col_idx] &= (
        features[:, col_idx] != -201.0
    )

print("Features shape:", features.shape)
print("Mask shape:", observed_mask.shape)
print("Mask dtype:", observed_mask.dtype)

print("Số giá trị được quan sát:",
      observed_mask.sum())

print("Số vị trí không có phép đo hợp lệ:",
      (~observed_mask).sum())

print("Số giá trị -201 được giữ nguyên:",
      np.sum(features == -201.0))


Features shape: (519, 25)
Mask shape: (519, 25)
Mask dtype: bool
Số giá trị được quan sát: 11903
Số vị trí không có phép đo hợp lệ: 1072
Số giá trị -201 được giữ nguyên: 1072


In [ ]:
import numpy as np

# Lấy timestamp từ MultiIndex của openSMILE
start_times = (
    features_df.index.get_level_values("start")
    .total_seconds()
    .to_numpy(dtype=np.float64)
)

end_times = (
    features_df.index.get_level_values("end")
    .total_seconds()
    .to_numpy(dtype=np.float64)
)

print("Start times shape:", start_times.shape)
print("End times shape:", end_times.shape)

print("\n5 thời điểm bắt đầu:")
print(start_times[:5])

print("\n5 thời điểm kết thúc:")
print(end_times[:5])

print("\nThời điểm bắt đầu đầu tiên:", start_times[0])
print("Thời điểm kết thúc cuối cùng:", end_times[-1])


Start times shape: (519,)
End times shape: (519,)

5 thời điểm bắt đầu:
[0.   0.01 0.02 0.03 0.04]

5 thời điểm kết thúc:
[0.02 0.03 0.04 0.05 0.06]

Thời điểm bắt đầu đầu tiên: 0.0
Thời điểm kết thúc cuối cùng: 5.2346875


In [ ]:
import numpy as np
import soundfile as sf

# Lấy thời lượng WAV
audio_duration = sf.info(str(wav_path)).duration

# 1. Kiểm tra số lượng timestamp
assert len(start_times) == features.shape[0]
assert len(end_times) == features.shape[0]

# 2. Kiểm tra timestamp là số hữu hạn
assert np.all(np.isfinite(start_times))
assert np.all(np.isfinite(end_times))

# 3. Kiểm tra thời gian hợp lệ
assert np.all(start_times >= 0)
assert np.all(end_times > start_times)

# 4. Kiểm tra thứ tự thời gian
assert np.all(np.diff(start_times) >= 0)
assert np.all(np.diff(end_times) >= 0)

# 5. Kiểm tra không vượt thời lượng WAV
tolerance = 1e-4
assert np.all(end_times <= audio_duration + tolerance)

print("Số cửa sổ:", len(start_times))
print("Thời lượng WAV:", audio_duration)
print("Thời điểm kết thúc cuối:", end_times[-1])
print("PASS: Toàn bộ timestamp hợp lệ!")


Số cửa sổ: 519
Thời lượng WAV: 5.2346875
Thời điểm kết thúc cuối: 5.2346875
PASS: Toàn bộ timestamp hợp lệ!


In [ ]:
# Đọc nội dung module hiện có
existing_code = module_path.read_text(encoding="utf-8")

# Tránh ghi đè hoặc thêm trùng hàm
assert "def prepare_audio(" in existing_code
assert "def extract_audio_features(" not in existing_code

feature_code = '''
import numpy as np
import opensmile

# Khởi tạo một lần và tái sử dụng
_SMILE = None


def get_smile():
    global _SMILE

    if _SMILE is None:
        _SMILE = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.LowLevelDescriptors
        )

    return _SMILE


def extract_audio_features(wav_path, config) -> dict:
    wav_path = Path(wav_path)

    if not wav_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy WAV: {wav_path}"
        )

    info = sf.info(str(wav_path))

    if info.duration <= 0:
        raise ValueError("WAV rỗng hoặc không hợp lệ")

    if info.samplerate != config["sample_rate"]:
        raise ValueError("Sample rate không đúng")

    if info.channels != 1 or info.subtype != "PCM_16":
        raise ValueError("WAV không đúng định dạng chung")

    # Trích xuất đặc trưng
    smile = get_smile()
    df = smile.process_file(str(wav_path))

    features = df.to_numpy(dtype=np.float32)
    feature_names = df.columns.tolist()

    if features.ndim != 2 or features.shape[0] == 0:
        raise ValueError("Không trích xuất được chuỗi đặc trưng")

    # Timestamp tính bằng giây
    start_times = (
        df.index.get_level_values("start")
        .total_seconds()
        .to_numpy(dtype=np.float64)
    )

    end_times = (
        df.index.get_level_values("end")
        .total_seconds()
        .to_numpy(dtype=np.float64)
    )

    # Kiểm tra timestamp
    if (
        len(start_times) != len(features)
        or len(end_times) != len(features)
        or not np.all(np.isfinite(start_times))
        or not np.all(np.isfinite(end_times))
        or not np.all(start_times >= 0)
        or not np.all(end_times > start_times)
        or not np.all(np.diff(start_times) >= 0)
        or not np.all(np.diff(end_times) >= 0)
        or not np.all(end_times <= info.duration + 1e-4)
    ):
        raise ValueError("Timestamp audio không hợp lệ")

    # Đánh dấu các giá trị hữu hạn
    observed_mask = np.isfinite(features)

    # Quy tắc thử nghiệm trên pilot:
    # -201 là giá trị sàn ở ba đặc trưng biên độ
    # tương đối F0, chưa tự thay đổi dữ liệu gốc.
    floor_features = [
        "F1amplitudeLogRelF0_sma3nz",
        "F2amplitudeLogRelF0_sma3nz",
        "F3amplitudeLogRelF0_sma3nz"
    ]

    for name in floor_features:
        if name in feature_names:
            col_idx = feature_names.index(name)
            observed_mask[:, col_idx] &= (
                features[:, col_idx] != -201.0
            )

    return {
        "features": features,
        "start_times": start_times,
        "end_times": end_times,
        "feature_names": feature_names,
        "observed_mask": observed_mask
    }
'''

# Giữ nguyên prepare_audio(), bổ sung hàm mới
module_path.write_text(
    existing_code + "\n\n" + feature_code,
    encoding="utf-8"
)

print("Đã cập nhật:", module_path)
print("Có prepare_audio:",
      "def prepare_audio(" in module_path.read_text())
print("Có extract_audio_features:",
      "def extract_audio_features(" in module_path.read_text())


Đã cập nhật: /content/drive/MyDrive/CMU-MOSI/src/preprocessing/audio_processor.py
Có prepare_audio: True
Có extract_audio_features: True


In [ ]:
import importlib
import preprocessing.audio_processor as audio_processor

# Tải lại phiên bản module vừa cập nhật
importlib.reload(audio_processor)

# Lấy hàm mới
extract_audio_features = audio_processor.extract_audio_features

print("Reload module thành công!")
print("Đã tìm thấy hàm extract_audio_features!")


Reload module thành công!
Đã tìm thấy hàm extract_audio_features!


In [ ]:
# Cấu hình chung của nhóm
config = {
    "sample_rate": 16000
}

# Gọi hàm trích xuất đặc trưng
audio_output = extract_audio_features(
    wav_path=wav_path,
    config=config
)

print("Trích xuất đặc trưng thành công!")

# Kiểm tra các thành phần đầu ra
for key, value in audio_output.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}, dtype={value.dtype}")
    else:
        print(f"{key}: {len(value)} phần tử")


Trích xuất đặc trưng thành công!
features: (519, 25), dtype=float32
start_times: (519,), dtype=float64
end_times: (519,), dtype=float64
feature_names: 25 phần tử
observed_mask: (519, 25), dtype=bool


In [ ]:
import numpy as np

# Lấy các thành phần đầu ra
f = audio_output["features"]
s = audio_output["start_times"]
e = audio_output["end_times"]
names = audio_output["feature_names"]
mask = audio_output["observed_mask"]

# Kiểm tra kích thước
assert f.ndim == 2
assert f.shape[0] == len(s) == len(e)
assert f.shape[1] == len(names)
assert mask.shape == f.shape

# Kiểm tra kiểu dữ liệu
assert f.dtype == np.float32
assert mask.dtype == np.bool_

# Kiểm tra timestamp
assert np.all(np.isfinite(s))
assert np.all(np.isfinite(e))
assert np.all(s >= 0)
assert np.all(e > s)
assert np.all(np.diff(s) >= 0)
assert np.all(np.diff(e) >= 0)

# Kiểm tra tên đặc trưng
assert len(names) == len(set(names))
assert names == list(smile.feature_names)

# Đối chiếu với kết quả đã chạy thủ công
assert np.allclose(f, features, equal_nan=True)
assert np.array_equal(mask, observed_mask)
assert np.allclose(s, start_times)
assert np.allclose(e, end_times)

print("PASS: Schema đầu ra hợp lệ!")
print("PASS: Kết quả khớp với quá trình xử lý thủ công!")
print("Số cửa sổ:", f.shape[0])
print("Số chiều đặc trưng:", f.shape[1])
print("Số vị trí mask=False:", (~mask).sum())


PASS: Schema đầu ra hợp lệ!
PASS: Kết quả khớp với quá trình xử lý thủ công!
Số cửa sổ: 519
Số chiều đặc trưng: 25
Số vị trí mask=False: 1072


In [ ]:
import numpy as np
import pandas as pd
import time

# Thư mục lưu đặc trưng pilot
FEATURE_DIR = ROOT / "preprocessing/audio/features_pilot"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

feature_results = []

for _, row in pilot.iterrows():
    clip_id = str(row["id"])
    wav_file = PILOT_WAV_DIR / f"{clip_id}.wav"

    start_time = time.perf_counter()

    try:
        # Gọi hàm trong audio_processor.py
        output = extract_audio_features(
            wav_path=wav_file,
            config={"sample_rate": 16000}
        )

        # Lưu đặc trưng theo ID
        feature_path = FEATURE_DIR / f"{clip_id}.npz"

        np.savez_compressed(
            feature_path,
            features=output["features"],
            start_times=output["start_times"],
            end_times=output["end_times"],
            feature_names=np.array(
                output["feature_names"],
                dtype=str
            ),
            observed_mask=output["observed_mask"]
        )

        elapsed = time.perf_counter() - start_time

        feature_results.append({
            "id": clip_id,
            "status": "success",
            "T_audio": output["features"].shape[0],
            "D_audio": output["features"].shape[1],
            "mask_false": int(
                (~output["observed_mask"]).sum()
            ),
            "processing_seconds": round(elapsed, 3),
            "feature_path": str(feature_path),
            "error": ""
        })

        print(f"PASS: {clip_id}")

    except Exception as e:
        elapsed = time.perf_counter() - start_time

        feature_results.append({
            "id": clip_id,
            "status": "failed",
            "T_audio": None,
            "D_audio": None,
            "mask_false": None,
            "processing_seconds": round(elapsed, 3),
            "feature_path": "",
            "error": str(e)
        })

        print(f"FAIL: {clip_id} - {e}")

# Tổng hợp kết quả
feature_report = pd.DataFrame(feature_results)

display(feature_report)

print(
    "Trích xuất thành công:",
    (feature_report["status"] == "success").sum(),
    "/",
    len(feature_report)
)


PASS: 7JsX8y1ysxY$_$12
PASS: 8OtFthrtaJM$_$23
PASS: 8d-gEyoeBzc$_$5
PASS: 9T9Hf74oK10$_$9
PASS: Bfr499ggo-0$_$12
PASS: I5y0__X72p0$_$35
PASS: I5y0__X72p0$_$6
PASS: Nzq88NnDkEk$_$32
PASS: VbQk4H8hgr0$_$24
PASS: Vj1wYRQjB-o$_$2


,id,status,T_audio,D_audio,mask_false,processing_seconds,feature_path,error
0,7JsX8y1ysxY$_$12,success,519,25,1072,0.223,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
1,8OtFthrtaJM$_$23,success,863,25,1249,0.354,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
2,8d-gEyoeBzc$_$5,success,556,25,1141,0.254,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
3,9T9Hf74oK10$_$9,success,575,25,601,0.251,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
4,Bfr499ggo-0$_$12,success,254,25,253,0.129,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
5,I5y0__X72p0$_$35,success,215,25,361,0.120,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
6,I5y0__X72p0$_$6,success,391,25,246,0.185,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
7,Nzq88NnDkEk$_$32,success,331,25,565,0.158,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
8,VbQk4H8hgr0$_$24,success,245,25,485,0.122,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,
9,Vj1wYRQjB-o$_$2,success,477,25,582,0.218,/content/drive/MyDrive/CMU-MOSI/preprocessing/...,


Trích xuất thành công: 10 / 10


In [ ]:

import numpy as np
import pandas as pd

validation_results = []

for _, row in pilot.iterrows():
    clip_id = str(row["id"])
    file_path = FEATURE_DIR / f"{clip_id}.npz"

    try:
        with np.load(file_path, allow_pickle=False) as data:
            required = {
                "features",
                "start_times",
                "end_times",
                "feature_names",
                "observed_mask"
            }

            assert required.issubset(set(data.files))

            f = data["features"]
            s = data["start_times"]
            e = data["end_times"]
            names = data["feature_names"]
            mask = data["observed_mask"]

            assert f.ndim == 2
            assert f.dtype == np.float32
            assert mask.dtype == np.bool_
            assert mask.shape == f.shape
            assert len(s) == len(e) == f.shape[0]
            assert len(names) == f.shape[1] == 25

            assert np.all(np.isfinite(s))
            assert np.all(np.isfinite(e))
            assert np.all(s >= 0)
            assert np.all(e > s)
            assert np.all(np.diff(s) >= 0)
            assert np.all(np.diff(e) >= 0)

            assert list(names) == list(smile.feature_names)

            wav_duration = sf.info(
                str(PILOT_WAV_DIR / f"{clip_id}.wav")
            ).duration

            assert np.all(e <= wav_duration + 1e-4)

            validation_results.append({
                "id": clip_id,
                "status": "pass",
                "T_audio": f.shape[0],
                "D_audio": f.shape[1],
                "error": ""
            })

    except Exception as ex:
        validation_results.append({
            "id": clip_id,
            "status": "failed",
            "T_audio": None,
            "D_audio": None,
            "error": str(ex)
        })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print(
    "Kiểm chứng thành công:",
    (validation_df["status"] == "pass").sum(),
    "/",
    len(validation_df)
)


,id,status,T_audio,D_audio,error
0,7JsX8y1ysxY$_$12,pass,519,25,
1,8OtFthrtaJM$_$23,pass,863,25,
2,8d-gEyoeBzc$_$5,pass,556,25,
3,9T9Hf74oK10$_$9,pass,575,25,
4,Bfr499ggo-0$_$12,pass,254,25,
5,I5y0__X72p0$_$35,pass,215,25,
6,I5y0__X72p0$_$6,pass,391,25,
7,Nzq88NnDkEk$_$32,pass,331,25,
8,VbQk4H8hgr0$_$24,pass,245,25,
9,Vj1wYRQjB-o$_$2,pass,477,25,


Kiểm chứng thành công: 10 / 10


In [ ]:
from pathlib import Path
import pandas as pd

REPORT_DIR = ROOT / "preprocessing/audio/reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Báo cáo kết quả trích xuất
feature_report.to_csv(
    REPORT_DIR / "pilot_feature_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# Báo cáo kiểm chứng các file NPZ
validation_df.to_csv(
    REPORT_DIR / "pilot_feature_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

# Nhật ký các mẫu lỗi hoặc chưa đạt kiểm chứng
failed_features = feature_report[
    feature_report["status"] != "success"
].copy()

failed_validation = validation_df[
    validation_df["status"] != "pass"
].copy()

failed_features.to_csv(
    REPORT_DIR / "pilot_feature_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

failed_validation.to_csv(
    REPORT_DIR / "pilot_validation_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu báo cáo tại:", REPORT_DIR)
print("Số clip trích xuất thành công:",
      (feature_report["status"] == "success").sum())
print("Số file kiểm chứng thành công:",
      (validation_df["status"] == "pass").sum())
print("Số lỗi trích xuất:", len(failed_features))
print("Số lỗi kiểm chứng:", len(failed_validation))


Đã lưu báo cáo tại: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/reports
Số clip trích xuất thành công: 10
Số file kiểm chứng thành công: 10
Số lỗi trích xuất: 0
Số lỗi kiểm chứng: 0


In [ ]:
import importlib.metadata
import subprocess

# Lấy phiên bản công cụ
opensmile_version = importlib.metadata.version("opensmile")
soundfile_version = importlib.metadata.version("soundfile")

ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True,
    check=True
).stdout.splitlines()[0]

# Nội dung tài liệu bàn giao
guide = f"""BÀN GIAO MODULE TIỀN XỬ LÝ AUDIO - CMU-MOSI
============================================

1. THÔNG TIN MODULE
Đường dẫn: src/preprocessing/audio_processor.py
Người phụ trách: Huy Hoàng
Phạm vi kiểm thử: 10 clip pilot

2. MÔI TRƯỜNG VÀ CÔNG CỤ
Môi trường: Google Colab
FFmpeg: {ffmpeg_version}
openSMILE: {opensmile_version}
soundfile: {soundfile_version}

Lệnh cài đặt:
pip install opensmile soundfile numpy

3. CẤU HÌNH XỬ LÝ AUDIO
Định dạng: WAV PCM 16-bit
Số kênh: 1 (mono)
Tần số lấy mẫu: 16000 Hz
Bộ đặc trưng: eGeMAPSv02
Mức trích xuất: LowLevelDescriptors (LLD)
Số chiều đặc trưng: 25

4. CÁC HÀM CHÍNH
prepare_audio(video_path, wav_path, config)
- Tách WAV từ video bằng FFmpeg.
- Kiểm tra định dạng và thời lượng WAV.
- Trả về đường dẫn file WAV.

extract_audio_features(wav_path, config)
- Trích xuất đặc trưng bằng openSMILE.
- Giữ thông tin thời gian của từng cửa sổ.
- Trả về dictionary gồm 5 thành phần.

5. CẤU TRÚC ĐẦU RA
features: (T_audio, 25), float32
start_times: (T_audio,), đơn vị giây
end_times: (T_audio,), đơn vị giây
feature_names: danh sách 25 tên đặc trưng
observed_mask: (T_audio, 25), kiểu bool

6. KẾT QUẢ CHẠY PILOT
Số clip pilot: 10
Tách WAV thành công: 10/10
Trích xuất đặc trưng thành công: 10/10
Kiểm chứng dữ liệu thành công: 10/10
Số lỗi thực thi ghi nhận: 0

7. LƯU Ý
- Không chuẩn hóa Z-score.
- Không lấy trung bình đặc trưng toàn clip.
- Không cắt hoặc padding chuỗi.
- Không cắt khoảng lặng hoặc thay đổi tốc độ audio.
- Giữ nguyên các giá trị đặc trưng gốc.
- Quy tắc mask đối với giá trị -201 ở ba
  đặc trưng F1/F2/F3 amplitudeLogRelF0
  đang áp dụng thử nghiệm trên pilot.
- Cần người tổng hợp xác nhận quy tắc mask
  trước khi áp dụng chính thức.
- Chưa chạy toàn bộ 2199 mẫu CMU-MOSI.

8. THƯ MỤC KẾT QUẢ
WAV: preprocessing/audio/wav/
Đặc trưng: preprocessing/audio/features_pilot/
Báo cáo: preprocessing/audio/reports/
"""

# Lưu tài liệu lên Google Drive
guide_path = REPORT_DIR / "audio_pilot_handover.txt"
guide_path.write_text(guide, encoding="utf-8")

print("Đã tạo tài liệu:", guide_path)
print("File tồn tại:", guide_path.exists())


Đã tạo tài liệu: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/reports/audio_pilot_handover.txt
File tồn tại: True


In [ ]:
from pathlib import Path

guide_path = (
    ROOT / "preprocessing/audio/reports/audio_pilot_handover.txt"
)

print("File tồn tại:", guide_path.exists())

if guide_path.exists():
    print("Kích thước:", guide_path.stat().st_size, "bytes")
    print("\nNội dung tài liệu:")
    print(guide_path.read_text(encoding="utf-8"))
else:
    print("Chưa tìm thấy tài liệu bàn giao.")


File tồn tại: True
Kích thước: 2231 bytes

Nội dung tài liệu:
BÀN GIAO MODULE TIỀN XỬ LÝ AUDIO - CMU-MOSI

1. THÔNG TIN MODULE
Đường dẫn: src/preprocessing/audio_processor.py
Người phụ trách: Huy Hoàng
Phạm vi kiểm thử: 10 clip pilot

2. MÔI TRƯỜNG VÀ CÔNG CỤ
Môi trường: Google Colab
FFmpeg: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
openSMILE: 2.6.0
soundfile: 0.14.0

Lệnh cài đặt:
pip install opensmile soundfile numpy

3. CẤU HÌNH XỬ LÝ AUDIO
Định dạng: WAV PCM 16-bit
Số kênh: 1 (mono)
Tần số lấy mẫu: 16000 Hz
Bộ đặc trưng: eGeMAPSv02
Mức trích xuất: LowLevelDescriptors (LLD)
Số chiều đặc trưng: 25

4. CÁC HÀM CHÍNH
prepare_audio(video_path, wav_path, config)
- Tách WAV từ video bằng FFmpeg.
- Kiểm tra định dạng và thời lượng WAV.
- Trả về đường dẫn file WAV.

extract_audio_features(wav_path, config)
- Trích xuất đặc trưng bằng openSMILE.
- Giữ thông tin thời gian của từng cửa sổ.
- Trả về dictionary gồm 5 thành phần.

5. CẤU TRÚC ĐẦU RA
features: (T_

In [ ]:

# Đọc tài liệu hiện tại
guide = guide_path.read_text(encoding="utf-8")

example = """

9. VÍ DỤ SỬ DỤNG MODULE

Thêm thư mục src vào Python path trước khi import.

from preprocessing.audio_processor import (
    prepare_audio,
    extract_audio_features
)

config = {"sample_rate": 16000}

# Bước 1: Tách WAV từ video
wav_path = prepare_audio(
    video_path="duong_dan_video.mp4",
    wav_path="duong_dan_audio.wav",
    config=config
)

# Bước 2: Trích xuất đặc trưng Audio
output = extract_audio_features(
    wav_path=wav_path,
    config=config
)

print(output["features"].shape)
print(output["start_times"].shape)
print(output["end_times"].shape)
print(len(output["feature_names"]))
print(output["observed_mask"].shape)

Lưu ý:
- Các đường dẫn trên là ví dụ, cần thay bằng đường dẫn thật.
- Module không tự chạy khi được import.
- Đầu ra giữ nguyên chuỗi thời gian, chưa Z-score hay padding.
"""

# Tránh thêm trùng khi chạy lại
if "9. VÍ DỤ SỬ DỤNG MODULE" not in guide:
    guide_path.write_text(
        guide + example,
        encoding="utf-8"
    )

print("Đã cập nhật tài liệu:", guide_path)
print("Có ví dụ sử dụng:",
      "9. VÍ DỤ SỬ DỤNG MODULE" in
      guide_path.read_text(encoding="utf-8"))


Đã cập nhật tài liệu: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/reports/audio_pilot_handover.txt
Có ví dụ sử dụng: True


In [1]:

try:
    print("ROOT:", ROOT)
    print("Số mẫu pilot:", len(pilot))
    print("Module Audio:", module_path)
    print("Colab vẫn giữ các biến cũ!")
except NameError:
    print("Colab đã reset, cần khởi tạo lại môi trường.")


Colab đã reset, cần khởi tạo lại môi trường.


In [ ]:

%pip install opensmile==2.6.0 soundfile==0.14.0 numpy pandas


In [4]:

from preprocessing.audio_processor import (
    prepare_audio,
    extract_audio_features
)

import opensmile

print("Import module Audio thành công!")
print("openSMILE:", opensmile.__version__)


Import module Audio thành công!
openSMILE: 2.6.0


In [5]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CMU-MOSI")

MANIFEST = ROOT / "preprocessing/shared/manifest.csv"
RAW_ZIP = ROOT / "Raw.zip"

# Đọc danh sách toàn bộ mẫu
manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "id": str,
        "video_id": str,
        "clip_id": str
    }
)

# Kiểm tra dữ liệu
print("Tổng số mẫu:", len(manifest))

print("\nSố mẫu theo split:")
print(manifest["split"].value_counts())

print("\nKiểm tra dữ liệu:")
print("ID trùng:", manifest["id"].duplicated().sum())
print("ID thiếu:", manifest["id"].isna().sum())
print("Đường dẫn video thiếu:",
      manifest["video_relpath"].isna().sum())

print("\nKiểm tra file nguồn:")
print("Raw.zip tồn tại:", RAW_ZIP.exists())

# Kiểm tra điều kiện bắt buộc
assert len(manifest) == 2199
assert manifest["id"].is_unique
assert manifest["id"].notna().all()
assert manifest["video_relpath"].notna().all()
assert RAW_ZIP.is_file()

print("\nPASS: Sẵn sàng kiểm tra 2.199 video!")


Tổng số mẫu: 2199

Số mẫu theo split:
split
train    1284
test      686
valid     229
Name: count, dtype: int64

Kiểm tra dữ liệu:
ID trùng: 0
ID thiếu: 0
Đường dẫn video thiếu: 0

Kiểm tra file nguồn:
Raw.zip tồn tại: True

PASS: Sẵn sàng kiểm tra 2.199 video!


In [6]:
import zipfile
import pandas as pd
from pathlib import PurePosixPath

with zipfile.ZipFile(RAW_ZIP, "r") as zf:
    # Lấy tất cả file thực sự có trong ZIP
    zip_files = {
        name for name in zf.namelist()
        if not name.endswith("/")
    }

# Chuẩn hóa dấu phân cách đường dẫn
manifest_paths = (
    manifest["video_relpath"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.removeprefix("./")
)

# Kiểm tra từng mẫu có video hay không
found_mask = manifest_paths.isin(zip_files)

found_count = int(found_mask.sum())
missing_count = int((~found_mask).sum())

print("Tổng số mẫu:", len(manifest))
print("Số video tìm thấy:", found_count)
print("Số video không tìm thấy:", missing_count)

if missing_count > 0:
    print("\nCác mẫu chưa tìm thấy video:")
    display(
        manifest.loc[
            ~found_mask,
            ["id", "video_relpath"]
        ].head(20)
    )
else:
    print("\nPASS: Tất cả 2.199 mẫu đều có video trong Raw.zip!")


Tổng số mẫu: 2199
Số video tìm thấy: 2199
Số video không tìm thấy: 0

PASS: Tất cả 2.199 mẫu đều có video trong Raw.zip!


In [7]:
from pathlib import Path
import pandas as pd

# Thư mục chạy toàn bộ dataset
FULL_DIR = ROOT / "preprocessing/audio_full"

FULL_WAV_DIR = FULL_DIR / "wav"
FULL_FEATURE_DIR = FULL_DIR / "features"
FULL_REPORT_DIR = FULL_DIR / "reports"

# Tạo thư mục nếu chưa tồn tại
for folder in [
    FULL_WAV_DIR,
    FULL_FEATURE_DIR,
    FULL_REPORT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Tạo bảng theo dõi trạng thái từng ID
STATUS_PATH = FULL_REPORT_DIR / "audio_full_status.csv"

if STATUS_PATH.exists():
    status_df = pd.read_csv(
        STATUS_PATH,
        dtype={"id": str}
    )
    print("Đã tìm thấy bảng trạng thái từ lần chạy trước.")
else:
    status_df = manifest[
        ["id", "video_id", "clip_id", "split"]
    ].copy()

    status_df["wav_status"] = "pending"
    status_df["feature_status"] = "pending"
    status_df["error"] = ""

    status_df.to_csv(
        STATUS_PATH,
        index=False,
        encoding="utf-8-sig"
    )
    print("Đã tạo bảng trạng thái mới.")

print("\nThư mục lưu WAV:", FULL_WAV_DIR)
print("Thư mục đặc trưng:", FULL_FEATURE_DIR)
print("Thư mục báo cáo:", FULL_REPORT_DIR)

print("\nTổng số ID theo dõi:", len(status_df))
print("File trạng thái tồn tại:", STATUS_PATH.exists())

assert len(status_df) == 2199
print("\nPASS: Đã chuẩn bị thư mục và bảng trạng thái!")


Đã tạo bảng trạng thái mới.

Thư mục lưu WAV: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio_full/wav
Thư mục đặc trưng: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio_full/features
Thư mục báo cáo: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio_full/reports

Tổng số ID theo dõi: 2199
File trạng thái tồn tại: True

PASS: Đã chuẩn bị thư mục và bảng trạng thái!


In [8]:

from pathlib import Path
import shutil

OLD_DIR = ROOT / "preprocessing/audio_full"
FULL_DIR = ROOT / "preprocessing/audio/audio_full"

# Chuyển thư mục cũ vào trong audio
if OLD_DIR.exists() and not FULL_DIR.exists():
    shutil.move(str(OLD_DIR), str(FULL_DIR))
    print("Đã chuyển audio_full vào trong audio.")
elif FULL_DIR.exists() and not OLD_DIR.exists():
    print("audio_full đã nằm đúng vị trí.")
elif OLD_DIR.exists() and FULL_DIR.exists():
    raise RuntimeError(
        "Cả hai thư mục đều tồn tại. Cần kiểm tra trước khi chuyển."
    )
else:
    FULL_DIR.mkdir(parents=True, exist_ok=True)
    print("Đã tạo thư mục audio_full mới.")

# Cập nhật các đường dẫn cho bước tiếp theo
FULL_WAV_DIR = FULL_DIR / "wav"
FULL_FEATURE_DIR = FULL_DIR / "features"
FULL_REPORT_DIR = FULL_DIR / "reports"

for folder in [FULL_WAV_DIR, FULL_FEATURE_DIR, FULL_REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

STATUS_PATH = FULL_REPORT_DIR / "audio_full_status.csv"

print("\nThư mục mới:", FULL_DIR)
print("Bảng trạng thái tồn tại:", STATUS_PATH.exists())
print("Số thư mục đầu ra:",
      len([FULL_WAV_DIR, FULL_FEATURE_DIR, FULL_REPORT_DIR]))

assert STATUS_PATH.exists(), "Chưa tìm thấy bảng trạng thái!"
print("\nPASS: Đã cập nhật đường dẫn thành công!")


Đã chuyển audio_full vào trong audio.

Thư mục mới: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full
Bảng trạng thái tồn tại: True
Số thư mục đầu ra: 3

PASS: Đã cập nhật đường dẫn thành công!


In [9]:
import zipfile
import tempfile
from pathlib import Path
import numpy as np

# Chọn clip đầu tiên trong manifest
sample = manifest.iloc[0]

sample_id = str(sample["id"])
video_relpath = str(sample["video_relpath"]).replace("\\", "/")

wav_path = FULL_WAV_DIR / f"{sample_id}.wav"
feature_path = FULL_FEATURE_DIR / f"{sample_id}.npz"

# Cấu hình WAV và trích xuất
config = {
    "sample_rate": 16000
}

# Giải nén tạm video từ Raw.zip
with tempfile.TemporaryDirectory() as temp_dir:
    temp_video = Path(temp_dir) / Path(video_relpath).name

    with zipfile.ZipFile(RAW_ZIP, "r") as zf:
        with zf.open(video_relpath) as source:
            with open(temp_video, "wb") as target:
                import shutil
                shutil.copyfileobj(source, target)

    # Tách WAV
    prepare_audio(temp_video, wav_path, config)

# Trích xuất đặc trưng
result = extract_audio_features(wav_path, config)

# Lưu kết quả
np.savez_compressed(
    feature_path,
    features=result["features"],
    start_times=result["start_times"],
    end_times=result["end_times"],
    feature_names=np.asarray(result["feature_names"], dtype=str),
    observed_mask=result["observed_mask"]
)

# Kiểm tra kết quả
assert wav_path.exists()
assert feature_path.exists()
assert result["features"].ndim == 2
assert result["features"].shape[1] == 25
assert result["observed_mask"].shape == result["features"].shape

print("ID:", sample_id)
print("WAV:", wav_path.exists())
print("Đặc trưng:", feature_path.exists())
print("Kích thước:", result["features"].shape)
print("Số tên đặc trưng:", len(result["feature_names"]))
print("\nPASS: Xử lý thử 1 clip thành công!")


ID: 03bSnISJMiM$_$11
WAV: True
Đặc trưng: True
Kích thước: (210, 25)
Số tên đặc trưng: 25

PASS: Xử lý thử 1 clip thành công!


In [10]:
import zipfile
import tempfile
import shutil
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

config = {"sample_rate": 16000}

# Dùng thư mục đã thiết lập ở bước 7.3
STATUS_PATH = FULL_REPORT_DIR / "audio_full_status.csv"

def valid_wav(path):
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        info = sf.info(str(path))
        return (
            info.samplerate == 16000
            and info.channels == 1
            and info.subtype == "PCM_16"
            and info.frames > 0
        )
    except Exception:
        return False

def valid_npz(path):
    if not path.is_file() or path.stat().st_size == 0:
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            keys = {
                "features", "start_times", "end_times",
                "feature_names", "observed_mask"
            }
            if not keys.issubset(data.files):
                return False

            features = data["features"]
            starts = data["start_times"]
            ends = data["end_times"]
            names = data["feature_names"]
            mask = data["observed_mask"]

            return bool(
                features.ndim == 2
                and features.shape[0] > 0
                and features.shape[1] == 25
                and features.dtype == np.float32
                and starts.shape == (features.shape[0],)
                and ends.shape == (features.shape[0],)
                and names.shape == (25,)
                and mask.shape == features.shape
                and mask.dtype == np.bool_
                and np.isfinite(starts).all()
                and np.isfinite(ends).all()
                and (starts >= 0).all()
                and (ends > starts).all()
                and (np.diff(starts) >= 0).all()
            )
    except Exception:
        return False

# Khởi tạo bảng trạng thái dựa trên manifest
status_df = manifest[
    ["id", "video_id", "clip_id", "split"]
].copy()

status_df["wav_status"] = "pending"
status_df["feature_status"] = "pending"
status_df["error"] = ""

# Đọc lại trạng thái cũ nếu có
if STATUS_PATH.exists():
    old = pd.read_csv(STATUS_PATH, dtype={"id": str})
    old = old.drop_duplicates("id", keep="last").set_index("id")

    for column in ["wav_status", "feature_status", "error"]:
        if column in old.columns:
            status_df[column] = (
                status_df["id"].map(old[column])
                .fillna(status_df[column])
            )

# Kiểm tra các file thực tế, không chỉ tin vào CSV
completed = 0
failed = 0

with zipfile.ZipFile(RAW_ZIP, "r") as zf:
    for idx, row in tqdm(
        manifest.iterrows(),
        total=len(manifest),
        desc="Audio CMU-MOSI"
    ):
        sample_id = str(row["id"])
        video_relpath = (
            str(row["video_relpath"])
            .replace("\\", "/")
            .removeprefix("./")
        )

        wav_path = FULL_WAV_DIR / f"{sample_id}.wav"
        feature_path = FULL_FEATURE_DIR / f"{sample_id}.npz"

        try:
            # Đã trích xuất hợp lệ thì bỏ qua
            if valid_npz(feature_path):
                status_df.at[idx, "feature_status"] = "success"
                status_df.at[idx, "wav_status"] = (
                    "success" if valid_wav(wav_path) else "missing"
                )
                status_df.at[idx, "error"] = ""
                completed += 1
                continue

            # Nếu chưa có WAV hợp lệ thì tách từ video
            if not valid_wav(wav_path):
                with tempfile.TemporaryDirectory() as temp_dir:
                    temp_video = (
                        Path(temp_dir) /
                        Path(video_relpath).name
                    )

                    with zf.open(video_relpath) as src:
                        with open(temp_video, "wb") as dst:
                            shutil.copyfileobj(src, dst)

                    prepare_audio(temp_video, wav_path, config)

            if not valid_wav(wav_path):
                raise ValueError("WAV không hợp lệ")

            status_df.at[idx, "wav_status"] = "success"

            # Trích xuất đặc trưng
            result = extract_audio_features(wav_path, config)

            # Lưu tạm trước, chỉ thay file chính khi hoàn tất
            temp_npz = feature_path.with_suffix(".tmp.npz")

            np.savez_compressed(
                temp_npz,
                features=result["features"],
                start_times=result["start_times"],
                end_times=result["end_times"],
                feature_names=np.asarray(
                    result["feature_names"], dtype=str
                ),
                observed_mask=result["observed_mask"]
            )

            if not valid_npz(temp_npz):
                raise ValueError("File NPZ không đạt kiểm tra")

            temp_npz.replace(feature_path)

            status_df.at[idx, "feature_status"] = "success"
            status_df.at[idx, "error"] = ""
            completed += 1

        except Exception as exc:
            failed += 1
            status_df.at[idx, "feature_status"] = "failed"
            status_df.at[idx, "error"] = (
                f"{type(exc).__name__}: {exc}"
            )

        finally:
            # Lưu tiến độ sau từng ID
            temp_status = STATUS_PATH.with_suffix(".tmp.csv")
            status_df.to_csv(
                temp_status,
                index=False,
                encoding="utf-8-sig"
            )
            temp_status.replace(STATUS_PATH)

print("\n===== KẾT QUẢ =====")
print("Tổng số ID:", len(manifest))
print("Số ID có NPZ hợp lệ:", completed)
print("Số ID lỗi trong lượt chạy:", failed)

print("\nTrạng thái trích xuất:")
print(status_df["feature_status"].value_counts())

print("\nBáo cáo:", STATUS_PATH)


Audio CMU-MOSI:   0%|          | 0/2199 [00:00<?, ?it/s]


===== KẾT QUẢ =====
Tổng số ID: 2199
Số ID có NPZ hợp lệ: 2199
Số ID lỗi trong lượt chạy: 0

Trạng thái trích xuất:
feature_status
success    2199
Name: count, dtype: int64

Báo cáo: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_status.csv


In [11]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

records = []
errors = []

for _, row in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Kiểm tra Audio"
):
    sample_id = str(row["id"])
    path = FULL_FEATURE_DIR / f"{sample_id}.npz"

    try:
        with np.load(path, allow_pickle=False) as data:
            features = data["features"]
            starts = data["start_times"]
            ends = data["end_times"]
            names = data["feature_names"]
            mask = data["observed_mask"]

            # Kiểm tra cấu trúc
            assert features.dtype == np.float32
            assert features.ndim == 2
            assert features.shape[1] == 25
            assert len(names) == 25
            assert starts.shape == (features.shape[0],)
            assert ends.shape == (features.shape[0],)
            assert mask.shape == features.shape
            assert mask.dtype == np.bool_

            # Kiểm tra timestamp
            assert np.isfinite(starts).all()
            assert np.isfinite(ends).all()
            assert (starts >= 0).all()
            assert (ends > starts).all()
            assert (np.diff(starts) >= 0).all()

            records.append({
                "id": sample_id,
                "split": row["split"],
                "frames": features.shape[0],
                "dimensions": features.shape[1],
                "nan_count": int(np.isnan(features).sum()),
                "inf_count": int(np.isinf(features).sum()),
                "minus201_count": int(
                    (features == -201).sum()
                ),
                "mask_false_count": int(
                    (~mask).sum()
                ),
                "last_end_time": float(ends[-1])
            })

    except Exception as exc:
        errors.append({
            "id": sample_id,
            "error": str(exc)
        })

# Lưu báo cáo
quality_df = pd.DataFrame(records)
error_df = pd.DataFrame(
    errors,
    columns=["id", "error"]
)

quality_path = (
    FULL_REPORT_DIR / "audio_full_quality_report.csv"
)
error_path = (
    FULL_REPORT_DIR / "audio_full_validation_errors.csv"
)

quality_df.to_csv(
    quality_path,
    index=False,
    encoding="utf-8-sig"
)

error_df.to_csv(
    error_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== KIỂM TRA CHẤT LƯỢNG =====")
print("Số mẫu hợp lệ:", len(quality_df))
print("Số mẫu lỗi:", len(error_df))

if not quality_df.empty:
    print("\nSố frame:")
    print(quality_df["frames"].describe())

    print("\nTổng số NaN:",
          quality_df["nan_count"].sum())
    print("Tổng số Inf:",
          quality_df["inf_count"].sum())
    print("Tổng số -201:",
          quality_df["minus201_count"].sum())
    print("Tổng số mask False:",
          quality_df["mask_false_count"].sum())

print("\nĐã lưu báo cáo:", quality_path)
print("Đã lưu nhật ký lỗi:", error_path)


Kiểm tra Audio:   0%|          | 0/2199 [00:00<?, ?it/s]


===== KIỂM TRA CHẤT LƯỢNG =====
Số mẫu hợp lệ: 2199
Số mẫu lỗi: 0

Số frame:
count    2199.000000
mean      421.652115
std       337.057254
min        17.000000
25%       208.000000
50%       324.000000
75%       535.000000
max      5240.000000
Name: frames, dtype: float64

Tổng số NaN: 0
Tổng số Inf: 0
Tổng số -201: 1172521
Tổng số mask False: 1172518

Đã lưu báo cáo: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_quality_report.csv
Đã lưu nhật ký lỗi: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_validation_errors.csv


In [12]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

mismatch_records = []

for _, row in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Kiểm tra -201 và mask"
):
    sample_id = str(row["id"])
    path = FULL_FEATURE_DIR / f"{sample_id}.npz"

    with np.load(path, allow_pickle=False) as data:
        features = data["features"]
        mask = data["observed_mask"]
        names = data["feature_names"]

        # -201 nhưng mask vẫn True
        positions = np.argwhere(
            (features == -201) & mask
        )

        for frame_idx, feature_idx in positions:
            mismatch_records.append({
                "id": sample_id,
                "frame": int(frame_idx),
                "feature_name": str(names[feature_idx]),
                "value": float(
                    features[frame_idx, feature_idx]
                ),
                "mask": bool(mask[frame_idx, feature_idx])
            })

mismatch_df = pd.DataFrame(
    mismatch_records,
    columns=[
        "id", "frame", "feature_name",
        "value", "mask"
    ]
)

print("\n===== KẾT QUẢ ĐỐI CHIẾU =====")
print("Số vị trí -201 nhưng mask=True:",
      len(mismatch_df))

if not mismatch_df.empty:
    print("\nChi tiết:")
    display(mismatch_df)

    print("\nThống kê theo đặc trưng:")
    print(
        mismatch_df["feature_name"].value_counts()
    )

# Lưu báo cáo
mismatch_path = (
    FULL_REPORT_DIR / "audio_full_mask_review.csv"
)

mismatch_df.to_csv(
    mismatch_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu:", mismatch_path)


Kiểm tra -201 và mask:   0%|          | 0/2199 [00:00<?, ?it/s]


===== KẾT QUẢ ĐỐI CHIẾU =====
Số vị trí -201 nhưng mask=True: 3

Chi tiết:


,id,frame,feature_name,value,mask
0,0h-zjBukYpk$_$25,1030,logRelF0-H1-H2_sma3nz,-201.0,True
1,0h-zjBukYpk$_$25,1030,logRelF0-H1-A3_sma3nz,-201.0,True
2,Sqr0AcuoNnk$_$17,49,logRelF0-H1-H2_sma3nz,-201.0,True



Thống kê theo đặc trưng:
feature_name
logRelF0-H1-H2_sma3nz    2
logRelF0-H1-A3_sma3nz    1
Name: count, dtype: int64

Đã lưu: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_mask_review.csv


In [13]:
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

duration_records = []
duration_errors = []

for _, row in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Kiểm tra thời lượng Audio"
):
    sample_id = str(row["id"])

    wav_path = FULL_WAV_DIR / f"{sample_id}.wav"
    npz_path = FULL_FEATURE_DIR / f"{sample_id}.npz"

    try:
        info = sf.info(str(wav_path))
        wav_duration = info.frames / info.samplerate

        with np.load(npz_path, allow_pickle=False) as data:
            starts = data["start_times"]
            ends = data["end_times"]

            first_start = float(starts[0])
            last_end = float(ends[-1])

        duration_diff = abs(
            wav_duration - last_end
        )

        duration_records.append({
            "id": sample_id,
            "wav_duration": wav_duration,
            "first_start": first_start,
            "last_end": last_end,
            "duration_diff": duration_diff,
            "within_tolerance": (
                first_start >= 0
                and first_start <= 0.02
                and duration_diff <= 0.05
            )
        })

    except Exception as exc:
        duration_errors.append({
            "id": sample_id,
            "error": str(exc)
        })

duration_df = pd.DataFrame(duration_records)
duration_error_df = pd.DataFrame(
    duration_errors,
    columns=["id", "error"]
)

# Lưu báo cáo
duration_path = (
    FULL_REPORT_DIR / "audio_full_duration_report.csv"
)

duration_error_path = (
    FULL_REPORT_DIR / "audio_full_duration_errors.csv"
)

duration_df.to_csv(
    duration_path,
    index=False,
    encoding="utf-8-sig"
)

duration_error_df.to_csv(
    duration_error_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== KIỂM TRA THỜI LƯỢNG =====")
print("Số WAV kiểm tra:", len(duration_df))
print("Số lỗi đọc:", len(duration_error_df))

if not duration_df.empty:
    print(
        "Số clip đạt ngưỡng:",
        duration_df["within_tolerance"].sum()
    )

    print(
        "Số clip vượt ngưỡng:",
        (~duration_df["within_tolerance"]).sum()
    )

    print(
        "Chênh lệch lớn nhất:",
        duration_df["duration_diff"].max(),
        "giây"
    )

    print("\nThống kê chênh lệch:")
    print(duration_df["duration_diff"].describe())

    if (~duration_df["within_tolerance"]).any():
        print("\nCác clip cần kiểm tra:")
        display(
            duration_df.loc[
                ~duration_df["within_tolerance"]
            ].head(20)
        )

print("\nĐã lưu báo cáo:", duration_path)


Kiểm tra thời lượng Audio:   0%|          | 0/2199 [00:00<?, ?it/s]


===== KIỂM TRA THỜI LƯỢNG =====
Số WAV kiểm tra: 2199
Số lỗi đọc: 0
Số clip đạt ngưỡng: 2199
Số clip vượt ngưỡng: 0
Chênh lệch lớn nhất: 1.000000082740371e-09 giây

Thống kê chênh lệch:
count    2.199000e+03
mean     6.366529e-12
std      7.955422e-11
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e-09
Name: duration_diff, dtype: float64

Đã lưu báo cáo: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_duration_report.csv


In [14]:

from pathlib import Path
from datetime import datetime

report_path = (
    FULL_REPORT_DIR / "audio_full_handover.txt"
)

report = f"""BÁO CÁO BÀN GIAO AUDIO PREPROCESSING - CMU-MOSI
================================================

1. THÔNG TIN
Người phụ trách: Huy Hoàng
Phạm vi: Toàn bộ 2.199 mẫu CMU-MOSI
Ngày lập: {datetime.now().strftime('%d/%m/%Y')}

2. CẤU HÌNH
FFmpeg: Tách WAV từ video gốc
WAV: PCM 16-bit, mono, 16 kHz
openSMILE: eGeMAPSv02 LowLevelDescriptors
Số chiều đặc trưng: 25
Dữ liệu chưa Z-score, chưa cắt hoặc padding.

3. KẾT QUẢ
Tổng số mẫu: {len(manifest)}
WAV kiểm tra: {len(duration_df)}
File đặc trưng hợp lệ: {len(quality_df)}
Số lỗi kiểm chứng: {len(error_df)}
Số lỗi đọc WAV: {len(duration_error_df)}

Số frame tối thiểu: {quality_df['frames'].min()}
Số frame trung bình: {quality_df['frames'].mean():.2f}
Số frame tối đa: {quality_df['frames'].max()}

Tổng số NaN: {quality_df['nan_count'].sum()}
Tổng số Inf: {quality_df['inf_count'].sum()}
Tổng số giá trị -201: {quality_df['minus201_count'].sum()}
Tổng số mask False: {quality_df['mask_false_count'].sum()}

Clip đạt kiểm tra thời lượng:
{int(duration_df['within_tolerance'].sum())}/{len(duration_df)}

4. CẤU TRÚC FILE NPZ
features: (T_audio, 25), float32
start_times: (T_audio,), đơn vị giây
end_times: (T_audio,), đơn vị giây
feature_names: 25 tên đặc trưng
observed_mask: (T_audio, 25), bool

5. VẤN ĐỀ CẦN THỐNG NHẤT
- Hiện tại giữ nguyên giá trị -201 trong features.
- Ba đặc trưng F1/F2/F3 amplitudeLogRelF0
  đang được đánh dấu mask=False tại giá trị -201.
- Có 3 vị trí -201 thuộc hai đặc trưng
  logRelF0-H1-H2 và logRelF0-H1-A3
  vẫn có mask=True.
- Cần người tổng hợp xác nhận quy tắc mask.
- Chưa chuẩn hóa Z-score.
- Chưa thực hiện alignment và padding.
- Kiểm tra thời lượng không thay thế kiểm chứng
  đồng bộ giữa audio và video.

6. THƯ MỤC BÀN GIAO
Module: src/preprocessing/audio_processor.py
WAV: preprocessing/audio/audio_full/wav/
Đặc trưng: preprocessing/audio/audio_full/features/
Báo cáo: preprocessing/audio/audio_full/reports/

7. FILE BÁO CÁO
audio_full_status.csv
audio_full_quality_report.csv
audio_full_validation_errors.csv
audio_full_mask_review.csv
audio_full_duration_report.csv
audio_full_duration_errors.csv
"""

report_path.write_text(report, encoding="utf-8")

print("Đã tạo báo cáo:", report_path)
print("File tồn tại:", report_path.exists())
print("Kích thước:", report_path.stat().st_size, "bytes")


Đã tạo báo cáo: /content/drive/MyDrive/CMU-MOSI/preprocessing/audio/audio_full/reports/audio_full_handover.txt
File tồn tại: True
Kích thước: 1996 bytes


In [15]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/CMU-MOSI")

module_path = ROOT / "src/preprocessing/audio_processor.py"

print("File tồn tại:", module_path.is_file())
print("Kích thước:", module_path.stat().st_size, "bytes")

print("\n===== NỘI DUNG MODULE =====")
print(module_path.read_text(encoding="utf-8"))


File tồn tại: True
Kích thước: 4420 bytes

===== NỘI DUNG MODULE =====

from pathlib import Path
import subprocess
import soundfile as sf


def prepare_audio(video_path, wav_path, config) -> Path:
    video_path = Path(video_path)
    wav_path = Path(wav_path)

    # 1. Kiểm tra video đầu vào
    if not video_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy video: {video_path}"
        )

    if video_path.stat().st_size == 0:
        raise ValueError("Video đầu vào rỗng")

    # 2. Tạo thư mục lưu WAV
    wav_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # 3. Tách audio bằng FFmpeg
    command = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vn",
        "-ac", "1",
        "-ar", str(config["sample_rate"]),
        "-c:a", "pcm_s16le",
        str(wav_path)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
       